In [ ]:
from tabulate import tabulate
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import svd, norm
from matplotlib import gridspec
import os
import sys

# VES_Schlumberger function 
def VES_Schlumberger(r, t, s):
    q = 13
    f = 10
    m = 4.438
    e = np.exp(0.5 * np.log(10) / m)
    h = 2 * q - 2
    u = s * np.exp(-f * np.log(10) / m)
    a = []
    for _ in range(1 + h):
        v = r[-1]
        for w in reversed(range(len(r) - 1)):
            aa = np.tanh(t[w] / u)
            v = (v + r[w] * aa) / (1 + v * aa / r[w])
        a.append(v)
        u *= e
    coeffs = [105, -262, 416, -746, 1065, -4390, 13396, -27841,
              16448, 8183, 2525, 336, 225]
    g = sum(c * a[i*2] for i, c in enumerate(coeffs)) / 10000
    return g

def jacobian_central_log(m, lr, x, par_frac=1e-4):
    """
    m: full model vector (r followed by t), positive entries
    lr: number of resistivity parameters
    x: AB/2 array
    par_frac: fractional perturbation for central diff, e.g. 1e-4
    returns J where J_ij = d[log(roa)]_i / d[log(m_j)]
    """
    ndata = len(x)
    npar = len(m)
    J = np.zeros((ndata, npar))
    # central difference on multiplicative perturbation (approx log-space)
    for j in range(npar):
        mj = m[j]
        # fractional step but ensure not zero
        delta = max(abs(mj * par_frac), 1e-12)
        m_plus = m.copy()
        m_minus = m.copy()
        m_plus[j] = mj + delta
        m_minus[j] = mj - delta

        # If subtraction would make negative or zero, do forward diff in log-space
        if m_minus[j] <= 0 or m_plus[j] <= 0:
            # forward step only
            m_minus[j] = mj
            m_plus[j] = mj + delta
            roa_plus = np.array([VES_Schlumberger(m_plus[:lr], m_plus[lr:], s) for s in x])
            roa_base = np.array([VES_Schlumberger(m[:lr], m[lr:], s) for s in x])
            denom = np.log(m_plus[j]) - np.log(mj)
            # guard denom
            if denom == 0:
                J[:, j] = 0.0
            else:
                J[:, j] = (np.log(roa_plus) - np.log(roa_base)) / denom
        else:
            roa_plus = np.array([VES_Schlumberger(m_plus[:lr], m_plus[lr:], s) for s in x])
            roa_minus = np.array([VES_Schlumberger(m_minus[:lr], m_minus[lr:], s) for s in x])
            denom = np.log(m_plus[j]) - np.log(m_minus[j])  # ~ 2*delta/mj
            if denom == 0:
                J[:, j] = 0.0
            else:
                J[:, j] = (np.log(roa_plus) - np.log(roa_minus)) / denom
    return J

# --- Weighted RMS function ---
def weighted_rms(roa_obs, roa_cal, rel_err=0.05):
    roa_obs = np.asarray(roa_obs)
    roa_cal = np.asarray(roa_cal)
    sigma = rel_err * np.maximum(roa_obs, 1e-12)
    weights = 1.0 / sigma
    num = np.sum((weights * (roa_obs - roa_cal)) ** 2)
    den = np.sum(weights ** 2)
    wrms = np.sqrt(num / den)
    wrms_percent = 100 * wrms / np.mean(roa_obs)
    return wrms, wrms_percent

# User settings / inputs
# Path to data (two-column: AB/2 (m), apparent resistivity (ohm.m))
# NOTE: Provide correct path or filename. Example: data_file = r"C:\Users\Harman\Desktop\Validation\Schlumberger Code\model 3a.dat"
data_file = r"C:\Users\Harman\Desktop\Inversion Code\Model 1.dat"

if not os.path.exists(data_file):
    print(f"ERROR: Data file not found: {data_file}")
    sys.exit(1)

data = np.loadtxt(data_file)
if data.ndim == 1 or data.shape[1] < 2:
    raise ValueError("Data file must have at least two columns: AB/2 and apparent resistivity")
x = data[:, 0]
roa_obs = data[:, 1]

# initial model: resistivities (ohm.m) then thicknesses (m)
r_init = np.array([100.0, 50.0, 20.0])
t_init = np.array([5.0, 10.0])  # last layer thickness not needed (infinite)

# inversion controls
maxiteration = 30
par = 1e-4            
lambda_reg = 0.1      
damping_init = 1e-2   
data_relerr = 0.05    
min_update = 1e-6     
verbose = True


# Initialization
r = r_init.copy()
t = t_init.copy()
m = np.concatenate((r, t))
lr = len(r)
lt = len(t)
npar = len(m)
ndata = len(x)
sigma = data_relerr * np.maximum(roa_obs, 1e-12)
W = np.diag(1.0 / sigma)

# Precompute weighting (diagonal) based on relative error estimate
sigma = data_relerr * np.maximum(roa_obs, 1e-12)
w_vec = 1.0 / sigma
W = np.diag(w_vec)

# storage for diagnostics: initialize all keys you will append to
history = {
    "iter": [],
    "nrms_log": [],
    "nrms_lin": [],
    "misfit_log": [],
    "lambda": []
}

# Build regularization operator L for first-order smoothness on log(resistivities)
# L size: (lr-1) x npar, acts on log(m)
if lr >= 2:
    L = np.zeros((lr - 1, npar))
    for i in range(lr - 1):
        L[i, i] = -1.0
        L[i, i + 1] = 1.0
else:
    L = np.zeros((0, npar))

# Inversion loop
iteration = 0
damping = damping_init

# initial predicted data and misfit (safety: ensure predicted > 0)
roa_pred = np.array([VES_Schlumberger(r, t, s) for s in x])
roa_pred = np.maximum(roa_pred, 1e-12)
e_log = np.log(np.maximum(roa_obs, 1e-12)) - np.log(roa_pred)
misfit_log = e_log @ e_log
nrms_log = 100.0 * norm(e_log) / norm(np.log(np.maximum(roa_obs, 1e-12)))

while iteration < maxiteration:
    iteration += 1
    r = m[:lr].copy()
    t = m[lr:].copy()

    # Jacobian in log-space (d log(roa)/ d log(m))
    A_log = jacobian_central_log(m, lr, x, par_frac=par)  # (ndata, npar)

    # Apply data weighting
    A_w = W @ A_log
    e_w = W @ e_log

    # Regularization: append lambda_reg * L rows and zeros in e
    if L.shape[0] > 0:
        A_aug = np.vstack([A_w, lambda_reg * L])
        e_aug = np.concatenate([e_w, np.zeros(L.shape[0])])
    else:
        A_aug = A_w.copy()
        e_aug = e_w.copy()

    
    # Explicit SVD: Z = U Λ V^T
    U, svals, Vt = svd(A_aug, full_matrices=False)
    Λ_vec = svals.copy()
    V = Vt.T

    success = False
    damping_try = damping
    inner_tries = 0
    max_inner = 8

    while inner_tries < max_inner and not success:
        inner_tries += 1

        # Damped pseudo-inverse constructed from SVD:
        # Z_plus = V diag( s / (s^2 + damping_try) ) U^T
        denom = (Λ_vec**2 + damping_try)
        # avoid zero denom
        denom = np.where(denom == 0, 1e-24, denom)
        inv_factors = Λ_vec / denom
        Λ_damped = np.diag(inv_factors)
        Z_plus = V @ Λ_damped @ U.T

        # parameter increment in log-space
        dmg = Z_plus @ e_aug
        # update multiplicatively to keep positivity
        m_trial = np.exp(np.log(np.maximum(m, 1e-12)) + dmg)

        # get predicted data for trial model
        r_trial = m_trial[:lr]
        t_trial = m_trial[lr:]
        roa_trial = np.array([VES_Schlumberger(r_trial, t_trial, s) for s in x])
        roa_trial = np.maximum(roa_trial, 1e-12)
        e_log_trial = np.log(np.maximum(roa_obs, 1e-12)) - np.log(roa_trial)
        misfit_log_trial = e_log_trial @ e_log_trial

        # accept if misfit decreased
        if misfit_log_trial <= misfit_log:
            # accept
            m = m_trial
            roa_pred = roa_trial
            e_log = e_log_trial
            misfit_log = misfit_log_trial
            success = True
            # slightly decrease damping for next iteration
            damping = max(damping_try * 0.7, 1e-12)
            
            break
        else:
            damping_try *= 10.0

    if not success:
        if verbose:
            print(f"Iter {iteration:02d}: no successful update after {max_inner} tries. Stopping.")
        break

    nrms_log = 100.0 * norm(e_log) / norm(np.log(np.maximum(roa_obs, 1e-12)))
    nrms_lin = 100.0 * norm(roa_pred - roa_obs) / norm(roa_obs)

    history["iter"].append(iteration)
    history["nrms_log"].append(nrms_log)
    history["nrms_lin"].append(nrms_lin)
    history["misfit_log"].append(misfit_log)
    history["lambda"].append(damping)

    # stopping criterion: small relative misfit improvement
    if iteration > 1:
        prev = history["misfit_log"][-2]
        curr = history["misfit_log"][-1]
        dfit = (prev - curr) / max(prev, 1e-12)
        if dfit < min_update:
            if verbose:
                print(f"Converged: relative misfit change {dfit:.3e} < {min_update}")
            break


# Final outputs
r_final = m[:lr].copy()
t_final = m[lr:].copy()
# compute cumulative depths for the defined thicknesses
depths = np.cumsum(t_final) if len(t_final) > 0 else np.array([])

roa_final = np.array([VES_Schlumberger(r_final, t_final, s) for s in x])
roa_final = np.maximum(roa_final, 1e-12)

nrms_log_final = 100.0 * norm(np.log(roa_final) - np.log(np.maximum(roa_obs, 1e-12))) / norm(np.log(np.maximum(roa_obs, 1e-12)))
nrms_lin_final = 100.0 * norm(roa_final - roa_obs) / norm(roa_obs)
rms_abs = norm(roa_final - roa_obs) / np.sqrt(len(roa_obs))
wrms, wrms_percent = weighted_rms(roa_obs, roa_final, rel_err=data_relerr)

# Layer table string
layer_data = []
cumulative_depth = 0.0
for i in range(len(r_final)):
    res = f"{r_final[i]:.2f}"
    if i < len(t_final):
        thick_val = t_final[i]
        cumulative_depth += thick_val
        thick = f"{thick_val:.2f}"
        depth = f"{cumulative_depth:.2f}"
    else:
        thick = "∞"
        depth = "∞"
    layer_data.append([i + 1, res, thick, depth])
layer_headers = ["Layer", "Resis(Ω·m)", "Thickness(m)", "Depth(m)"]
layer_table_str = tabulate(layer_data, headers=layer_headers, tablefmt="grid")


# Plotting: Apparent resistivity + layer table
fig = plt.figure(figsize=(12, 6))
gs = gridspec.GridSpec(1, 1)

# Apparent Resistivity Curve
ax = plt.subplot(gs[0])
ax.loglog(x, roa_obs, 'r.-', label='Field Data')
ax.loglog(x, roa_final, 'k.-', label='Calculated Data')

# Step model overlay (extended last layer)
ab2_edges = []
resist_values = []
extension_length = max(x) * 1.5
for i in range(len(r_final)):
    if i == 0:
        top = min(x)
    else:
        top = depths[i - 1]
    bottom = depths[i] if i < len(depths) else extension_length
    ab2_edges += [top, bottom]
    resist_values += [r_final[i], r_final[i]]

ax.loglog(ab2_edges, resist_values, linestyle='--', linewidth=1, label='Final Model (Step)')

# Fixed limits
ax.set_xlim(1, 1000)   # AB/2 from 1 to 1000 m
ax.set_ylim(1, 1000)   # Apparent resistivity from 1 to 1000 Ω·m

# 
ax.set_xlabel('Current Electrode Position (AB/2) (m)', fontsize=12)
ax.set_ylabel('Apparent Resistivity (Ω·m)', fontsize=12)
ax.set_title("Schlumberger Configuration", fontsize=13)
ax.grid(True, which='both', linestyle='--', linewidth=0.5)
ax.legend(loc='lower right', fontsize=10)

# RMS and Iteration info 
ax.text(0.02, 0.98, f'NRMS-Error (log) = {nrms_log_final:.2f}%', transform=ax.transAxes,
         fontsize=12, color='blue', ha='left', va='top')
ax.text(0.02, 0.93, f'NRMS-Error (lin) = {nrms_lin_final:.2f}%', transform=ax.transAxes,
         fontsize=12, color='Red', ha='left', va='top')
ax.text(0.02, 0.83, f'Model Iteration: {iteration}',
        transform=ax.transAxes, fontsize=12, color='darkgreen',
        ha='left', va='top')
ax.text(0.5, 1.05, " Model 1 ", transform=ax.transAxes,
         fontsize=12, color='black', ha='center', va='bottom')
ax.text(0.02, 0.86, f"Weighted RMS = {wrms_percent:.2f}%", transform=ax.transAxes, fontsize=12, color='purple')

# Layer model table on the right side 
table_text = "Layer Model Table\n" + layer_table_str

# Position it to the right of the plotting area
ax.text(1.05, 0.65, table_text, transform=ax.transAxes,
        fontsize=10, family='monospace', va='bottom', ha='left',
        bbox=dict(facecolor='white', alpha=0.85, boxstyle='round,pad=0.5'))

plt.tight_layout()
plt.savefig("Synthetic Model 1.png", dpi=300, bbox_inches='tight')
plt.show()

# AB/2 table
ab_data = np.column_stack((x, np.round(roa_obs, 2), np.round(roa_final, 2)))
ab_headers = ["AB/2 (m)", "Field Data (Ω·m)", "Calculated Data (Ω·m)"]
ab_table_str = tabulate(ab_data, headers=ab_headers, tablefmt="grid")

# Print results
print("\nLayer Model Table:")
print(layer_table_str)
print("\nLayer Resistivities:", np.round(r_final, 2))
print("Final Layer Thicknesses:", np.round(t_final, 2))
print(f"NRMS-Error (log-domain) = {nrms_log_final:.2f}%")
print(f"NRMS-Error (linear) = {nrms_lin_final:.2f}%")
print(f"Weighted RMS: {wrms_percent:.2f}%")

with open("S1_Table.txt", "w", encoding="utf-8") as f:
    f.write("Layer Model Table:\n")
    f.write(layer_table_str)

with open("S1.txt", "w", encoding="utf-8") as f:
    f.write("AB/2 Field Data vs Calculated Data Table:\n")
    f.write(ab_table_str)